# Configurable Runnable Reference

Developer-facing classes and functions defined in `langchain_core.runnables.configurable`.

# `prefix_config_spec`

Adds a namespace prefix to the identifier of a non-shared configuration specification.

Shared specifications are returned unchanged.

```python
prefix_config_spec(
    spec: ConfigurableFieldSpec, # Configuration specification to process
    prefix: str, # Prefix added before the specification identifier
) -> ConfigurableFieldSpec # Return the prefixed or unchanged specification
```

---

# `make_options_spec`

Creates a `ConfigurableFieldSpec` from a single-option or multiple-option configurable field.

```python
make_options_spec(
    spec: ConfigurableFieldSingleOption | ConfigurableFieldMultiOption, # Option-based configurable field definition
    description: str | None, # Fallback description used when the field has no description
) -> ConfigurableFieldSpec # Return the generated configuration specification
```

For a single-option field, the generated annotation is a string enum.

For a multiple-option field, the generated annotation is a sequence of string-enum values.

# `DynamicRunnable: RunnableSerializable[Input, Output]`

`DynamicRunnable` is an abstract serializable wrapper that selects or rebuilds a `Runnable` from runtime configuration before execution.

It is normally created through `configurable_fields()` or `configurable_alternatives()` rather than instantiated directly.

## Fields

```python
default: RunnableSerializable[Input, Output] # Default Runnable used when configuration does not replace it
config: RunnableConfig | None = None # Configuration permanently associated with the wrapper
```

## Constructor

```python
DynamicRunnable(
    *,
    default: RunnableSerializable[Input, Output], # Default wrapped Runnable
    config: RunnableConfig | None = None, # Optional stored runtime configuration
    name: str | None = None, # Optional Runnable name
) -> None # Initialize the dynamic Runnable
```

## Public Method

### `prepare`

Resolves nested dynamic wrappers and returns the final Runnable together with the merged configuration.

```python
prepare(
    self, # DynamicRunnable instance
    config: RunnableConfig | None = None, # Optional call-time configuration
) -> tuple[Runnable[Input, Output], RunnableConfig] # Return the prepared Runnable and merged configuration
```

## Overridden Properties and Methods

### `is_lc_serializable`

Returns `True` because dynamic Runnables support LangChain serialization.

### `get_lc_namespace`

Returns the LangChain Runnable serialization namespace.

### `InputType`

Returns the input type of the default Runnable.

### `OutputType`

Returns the output type of the default Runnable.

### `get_input_schema`

Prepares the selected Runnable and returns its input schema.

### `get_output_schema`

Prepares the selected Runnable and returns its output schema.

### `get_graph`

Prepares the selected Runnable and returns its execution graph.

### `with_config`

Returns a new dynamic wrapper containing the supplied configuration.

### `invoke`

Prepares and synchronously invokes the selected Runnable.

### `ainvoke`

Prepares and asynchronously invokes the selected Runnable.

### `batch`

Prepares the appropriate Runnable for every input and executes synchronous batch processing.

### `abatch`

Prepares the appropriate Runnable for every input and executes asynchronous batch processing.

### `stream`

Prepares the selected Runnable and synchronously streams its output.

### `astream`

Prepares the selected Runnable and asynchronously streams its output.

### `transform`

Prepares the selected Runnable and transforms a synchronous input iterator.

### `atransform`

Prepares the selected Runnable and transforms an asynchronous input iterator.

### `__getattr__`

Delegates missing attributes and methods to the default or currently selected Runnable.

In [1]:
# its abstract class so no need to worry about this
from typing import Any, Literal # Import types used for annotations

from langchain_core.runnables import Runnable, RunnableConfig, RunnableSerializable # Import Runnable base types
from langchain_core.runnables.config import ensure_config # Import configuration normalizer
from langchain_core.runnables.configurable import DynamicRunnable # Import the abstract DynamicRunnable class


class TextCaseRunnable(RunnableSerializable[str, str]): # Create a concrete serializable Runnable
    mode: Literal["upper", "lower"] # Restrict the mode to supported string values

    def invoke( # Implement the abstract Runnable execution method
        self, # Current TextCaseRunnable instance
        input: str, # Input text received by the Runnable
        config: RunnableConfig | None = None, # Optional runtime configuration
        **kwargs: Any, # Additional execution arguments
    ) -> str: # Return the converted string
        return input.upper() if self.mode == "upper" else input.lower() # Convert text according to the stored mode


class CaseSelector(DynamicRunnable[str, str]): # Create a concrete subclass of the abstract DynamicRunnable class
    upper_runnable: RunnableSerializable[str, str] # Store the uppercase Runnable alternative

    def _prepare( # Implement the abstract method required by DynamicRunnable
        self, # Current CaseSelector instance
        config: RunnableConfig | None = None, # Optional runtime configuration
    ) -> tuple[Runnable[str, str], RunnableConfig]: # Return the selected Runnable and normalized configuration
        normalized_config: RunnableConfig = ensure_config(config) # Normalize the supplied configuration

        selected_mode: str = normalized_config["configurable"].get( # Read the configured mode
            "case", # Use the case configuration key
            "lower", # Use lowercase as the default mode
        ) # Finish reading the configured mode

        if selected_mode == "upper": # Check whether the uppercase alternative was selected
            return self.upper_runnable, normalized_config # Return the uppercase Runnable

        return self.default, normalized_config # Return the default lowercase Runnable


lower_runnable: TextCaseRunnable = TextCaseRunnable( # Create the default concrete Runnable
    mode="lower", # Configure it to produce lowercase text
) # Finish creating the lowercase Runnable

upper_runnable: TextCaseRunnable = TextCaseRunnable( # Create the alternative concrete Runnable
    mode="upper", # Configure it to produce uppercase text
) # Finish creating the uppercase Runnable

dynamic_runnable: CaseSelector = CaseSelector( # Instantiate the concrete DynamicRunnable subclass
    default=lower_runnable, # Set the default Runnable inherited from DynamicRunnable
    upper_runnable=upper_runnable, # Set the uppercase alternative
) # Finish creating the dynamic selector

default_result: str = dynamic_runnable.invoke( # Execute without alternative configuration
    "Hello LangChain", # Pass the input text
) # Use the default lowercase Runnable

upper_result: str = dynamic_runnable.invoke( # Execute with runtime configuration
    "Hello LangChain", # Pass the input text
    config={ # Provide Runnable runtime configuration
        "configurable": { # Supply values used for dynamic selection
            "case": "upper", # Select the uppercase Runnable
        }, # Finish configurable values
    }, # Finish runtime configuration
) # Finish the configured execution

print(default_result) # Display the default lowercase result

print(upper_result) # Display the configured uppercase result

hello langchain
HELLO LANGCHAIN


# `RunnableConfigurableFields: DynamicRunnable[Input, Output]`

`RunnableConfigurableFields` allows selected model fields of a serializable Runnable to be replaced through `config["configurable"]`.

It is normally produced by calling `configurable_fields()` on a `RunnableSerializable`.

## Fields

```python
default: RunnableSerializable[Input, Output] # Default Runnable whose fields may be changed
config: RunnableConfig | None = None # Configuration permanently associated with the wrapper
fields: dict[str, AnyConfigurableField] # Model field names mapped to configurable-field definitions
```

## Constructor

```python
RunnableConfigurableFields(
    *,
    default: RunnableSerializable[Input, Output], # Runnable containing the configurable model fields
    fields: dict[str, AnyConfigurableField], # Field names mapped to configuration specifications
    config: RunnableConfig | None = None, # Optional stored runtime configuration
    name: str | None = None, # Optional Runnable name
) -> None # Initialize the configurable-fields wrapper
```

## Overridden Properties and Methods

### `config_specs`

Returns configuration specifications for all declared configurable fields together with specifications exposed by the default Runnable.

It supports regular configurable fields, single-choice options, and multiple-choice options.

### `configurable_fields`

Returns a new wrapper containing the existing field definitions merged with newly supplied definitions.

## Behaviour

- Runtime values are read using each configurable field's `id`.
- Regular configurable values replace matching model fields.
- Single-option values select one value from the configured options mapping.
- Multi-option values select multiple values from the configured options mapping.
- The default Runnable is reconstructed only when configurable values are supplied.
- Unrecognized configurable keys are ignored by this wrapper.

In [2]:
from langchain_core.language_models.fake import FakeListLLM # Import a concrete serializable fake LLM
from langchain_core.runnables import ConfigurableField # Import configurable-field definition
from langchain_core.runnables.configurable import RunnableConfigurableFields # Import the concrete configurable class

default_model: FakeListLLM = FakeListLLM( # Create the underlying concrete Runnable
    responses=["Default response"] # Set the default list of model responses
) # Finish creating the fake model

configurable_model: RunnableConfigurableFields = RunnableConfigurableFields( # Instantiate the concrete class directly
    default=default_model, # Set the Runnable whose field can be changed
    fields={ # Declare the configurable model fields
        "responses": ConfigurableField( # Make the responses field configurable
            id="model_responses", # Set the runtime configuration key
            name="Model Responses", # Set a readable field name
            description="Responses returned by the fake model", # Describe the configurable field
            annotation=list[str], # Specify the expected runtime value type
        ) # Finish defining the configurable field
    }, # Finish the configurable-fields mapping
) # Finish creating RunnableConfigurableFields

default_result: str = configurable_model.invoke( # Execute with the original field value
    "Any prompt" # Pass an input prompt
) # Finish the default execution

configured_result: str = configurable_model.invoke( # Execute with a replacement field value
    "Any prompt", # Pass an input prompt
    config={ # Supply runtime configuration
        "configurable": { # Supply values for configurable fields
            "model_responses": ["Configured response"] # Replace the responses field for this call
        } # Finish configurable values
    } # Finish runtime configuration
) # Finish the configured execution

print(default_result) # Display the default response
print(configured_result) # Display the configured response

Default response
Configured response


# `RunnableConfigurableAlternatives: DynamicRunnable[Input, Output]`

`RunnableConfigurableAlternatives` selects one Runnable implementation from a default Runnable and a named alternatives mapping.

The selected key is read from `config["configurable"]` using the identifier stored in `which`.

## Fields

```python
default: RunnableSerializable[Input, Output] # Runnable used when the default key is selected
config: RunnableConfig | None = None # Configuration permanently associated with the wrapper
which: ConfigurableField # Configuration field used to choose an alternative
alternatives: dict[str, Runnable[Input, Output] | Callable[[], Runnable[Input, Output]]] # Named Runnables or lazy Runnable factories
default_key: str = "default" # Configuration value selecting the default Runnable
prefix_keys: bool # Whether nested configurable keys are namespaced by alternative
```

## Constructor

```python
RunnableConfigurableAlternatives(
    *,
    which: ConfigurableField, # Configuration field controlling alternative selection
    default: RunnableSerializable[Input, Output], # Default Runnable implementation
    alternatives: dict[str, Runnable[Input, Output] | Callable[[], Runnable[Input, Output]]], # Alternative keys mapped to Runnables or lazy factories
    prefix_keys: bool, # Whether alternative-specific configuration keys use prefixes
    default_key: str = "default", # Key representing the default Runnable
    config: RunnableConfig | None = None, # Optional stored runtime configuration
    name: str | None = None, # Optional Runnable name
) -> None # Initialize the configurable-alternatives wrapper
```

## Overridden Properties and Methods

### `config_specs`

Returns the alternative-selection specification together with configuration specifications exposed by the default and serializable alternative Runnables.

When `prefix_keys` is enabled, alternative-specific specification identifiers are namespaced.

### `configurable_fields`

Returns a new alternatives wrapper whose default Runnable contains the supplied configurable-field definitions.

## Behaviour

- The default Runnable is selected when no alternative key is supplied.
- A stored Runnable alternative is returned directly.
- A callable alternative is invoked lazily only when selected.
- An unknown alternative key raises `ValueError`.
- With `prefix_keys=True`, keys follow the form `field_id==alternative_key/configuration_id`.
- Shared configuration specifications are not prefixed.


In [ ]:
from langchain_core.language_models.fake import FakeListLLM # Import a concrete serializable fake model
from langchain_core.runnables import ConfigurableField # Import the alternative-selection field
from langchain_core.runnables.configurable import RunnableConfigurableAlternatives # Import the concrete configurable class


def create_formal_model() -> FakeListLLM: # Define a lazy factory for one alternative
    return FakeListLLM(responses=["Good day. How may I assist you?"]) # Create the model only when selected


default_model: FakeListLLM = FakeListLLM( # Create the default model
    responses=["Hello! How can I help?"] # Configure its fixed response
) # Finish creating the default model

friendly_model: FakeListLLM = FakeListLLM( # Create a stored alternative model
    responses=["Hey! What can I do for you?"] # Configure its friendly response
) # Finish creating the friendly model

configurable_model: RunnableConfigurableAlternatives[str, str] = RunnableConfigurableAlternatives( # Create the alternatives wrapper
    which=ConfigurableField( # Define the runtime selection field
        id="response_style", # Set the key used inside config["configurable"]
        name="Response Style", # Set a readable configuration name
        description="Select the response style", # Explain what the field controls
        annotation=str, # Specify that the selected key must be a string
    ), # Finish defining the selection field
    default=default_model, # Set the model used when no alternative is selected
    alternatives={ # Define the available alternative implementations
        "friendly": friendly_model, # Store an already-created Runnable alternative
        "formal": create_formal_model, # Store a lazy factory called only when selected
    }, # Finish defining the alternatives
    default_key="default", # Set the key representing the default model
    prefix_keys=False, # Keep nested configuration keys without alternative prefixes
) # Finish creating RunnableConfigurableAlternatives

default_result: str = configurable_model.invoke( # Invoke without selecting an alternative
    "Help me" # Pass the input prompt
) # Use the default model

friendly_result: str = configurable_model.invoke( # Invoke the stored alternative
    "Help me", # Pass the input prompt
    config={ # Supply runtime configuration
        "configurable": { # Supply configurable values
            "response_style": "friendly", # Select the friendly alternative
        } # Finish configurable values
    } # Finish runtime configuration
) # Finish the friendly invocation

formal_result: str = configurable_model.invoke( # Invoke the lazy alternative
    "Help me", # Pass the input prompt
    config={ # Supply runtime configuration
        "configurable": { # Supply configurable values
            "response_style": "formal", # Select the formal alternative
        } # Finish configurable values
    } # Finish runtime configuration
) # Finish the formal invocation

print(default_result) # Display the default response

print(friendly_result) # Display the friendly response

print(formal_result) # Display the formal response